In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import os

In [2]:
dataset_path = "dataset/train"

if not os.path.exists(dataset_path):
    raise FileNotFoundError("Dataset folder NOT found")

print("Dataset folder found ✔")

Dataset folder found ✔


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cpu


In [5]:
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

In [6]:
train_dataset = datasets.ImageFolder(
    root=dataset_path,
    transform=transform
)

print("Classes Found:", train_dataset.classes)
print("Total Images:", len(train_dataset))

Classes Found: ['AbnormalHB', 'Normal']
Total Images: 517


In [7]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0
)

In [8]:
model = models.resnet18(pretrained=True)


C:\Users\Shubham\AppData\Roaming\Python\Python313\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\Shubham\AppData\Roaming\Python\Python313\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [9]:
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, len(train_dataset.classes))

model = model.to(device)

In [10]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [11]:
# num_epochs = 5

# for epoch in range(num_epochs):

#     model.train()
#     running_loss = 0
#     correct = 0
#     total = 0

#     for images, labels in train_loader:

#         images = images.to(device)
#         labels = labels.to(device)

#         # Forward pass
#         outputs = model(images)
#         loss = criterion(outputs, labels)

#         # Backpropagation
#         optimizer.zero_grad()
#         loss.backward()
#         optimizer.step()

#         running_loss += loss.item()

#         # Accuracy calculation
#         _, predicted = torch.max(outputs, 1)
#         total += labels.size(0)
#         correct += (predicted == labels).sum().item()

#     print(f"\nEpoch {epoch+1}/{num_epochs}")
#     print("Loss:", running_loss / len(train_loader))
#     print("Accuracy:", 100 * correct / total)
num_epochs = 5

for epoch in range(num_epochs):

    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()


KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), "ecg_model.pth")
print("Model Saved Successfully ✔")

Model Saved Successfully ✔


In [ ]:
from PIL import Image

model.eval()

def predict_ecg(image_path):

    img = Image.open(image_path).convert('RGB')
    img = transform(img)
    img = img.unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(img)
        _, predicted = torch.max(output, 1)

    classes = ['AbnormalHB', 'Normal']

    print("Prediction:", classes[predicted.item()])



In [ ]:
predict_ecg("dataset/train/Normal/Normal(1).jpg")
predict_ecg("dataset/train/AbnormalHB/HB(6).jpg")



Prediction: Normal
Prediction: AbnormalHB


In [ ]:
from collections import Counter

labels = [label for _, label in train_dataset]
print(Counter(labels))


Counter({1: 284, 0: 233})


In [ ]:
print(train_dataset.class_to_idx)


{'AbnormalHB': 0, 'Normal': 1}


In [49]:
print(device)


cpu
